In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [ ]:
# read the email.csv file into a DataFrame
data = pd.read_csv('email.csv')

# display the first 10 rows of the data
print("First 10 rows of the dataset:")
print(data.head(10))

First 10 rows of the dataset:
  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
5     spam  FreeMsg Hey there darling it's been 3 week's n...
6      ham  Even my brother is not like to speak with me. ...
7      ham  As per your request 'Melle Melle (Oru Minnamin...
8     spam  WINNER!! As a valued network customer you have...
9     spam  Had your mobile 11 months or more? U R entitle...


In [ ]:
print("First 10 entries of the 'Message' column:")
print(data['Message'].head(10))

First 10 entries of the 'Message' column:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
5    FreeMsg Hey there darling it's been 3 week's n...
6    Even my brother is not like to speak with me. ...
7    As per your request 'Melle Melle (Oru Minnamin...
8    WINNER!! As a valued network customer you have...
9    Had your mobile 11 months or more? U R entitle...
Name: Message, dtype: object


In [ ]:
# initialize lemmatizer and load stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# I used a function, because I'll use the same preprocessing on every message I have
def preprocessing(text):
    # remove punctuation and marks using regex
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\d+', '', text) # I don't if I should remove numbers or not

    # convert to lowercase (normalization)
    text = text.lower()

    # tokenize the text using word tokenizer
    words = nltk.word_tokenize(text)

    # remove Stop words
    words = [word for word in words if word not in stop_words]

    # use lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    # Join the words back into a single string
    return " ".join(words)

# apply preprocessing to all messages in the data
data['clean_messages'] = data['Message'].apply(preprocessing)

print("Original Message vs. clean Message:")
print(data[['Message', 'clean_messages']].head(10))

Original Message vs. clean Message:
                                             Message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   
5  FreeMsg Hey there darling it's been 3 week's n...   
6  Even my brother is not like to speak with me. ...   
7  As per your request 'Melle Melle (Oru Minnamin...   
8  WINNER!! As a valued network customer you have...   
9  Had your mobile 11 months or more? U R entitle...   

                                      clean_messages  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  
3                u dun say early hor u c already say  
4           nah dont think go usf life around though  
5  freemsg hey da

In [ ]:
data['Category'] = data['Category'].str.strip().str.lower()

data['Category'] = data['Category'].map({'ham': 0,'spam': 1})

In [ ]:
data.dropna(subset=['Category', 'clean_messages'], inplace=True)

In [ ]:
data.isnull().sum()

,0
Category,0
Message,0
clean_messages,0


In [ ]:
data.head()

,Category,Message,clean_messages
0,0.0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0.0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1.0,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,0.0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0.0,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though


In [ ]:
data['Category'] = data['Category'].astype(int)

In [ ]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(data['clean_messages'])
y = data['Category']

print("TF-IDF Shape:", X.shape)

TF-IDF Shape: (5572, 7946)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 4457
Testing samples: 1115


In [ ]:


rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9739910313901345

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       1.00      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.99      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115


Confusion Matrix:
[[966   0]
 [ 29 120]]
